
# **Step 1:** The Ingestion Block (Connecting & Extracting)

This block sets up a secure connection socket to your locally running MetaTrader 5 terminal, requests a clean slice of historical time-series data, and converts the raw binary structure into a localized Pandas DataFrame.
Create a new cell and write the following code:

In [14]:
import MetaTrader5 as mt5 
import pandas as pd 
import numpy as np

def connect_and_fetch_forex_data(symbol:str, timeframe,num_bars:int) -> pd.DataFrame:
    if not mt5. initialize():
        raise RuntimeError(f"Could not Connect to MT5. \n Error: {mt5.last_error()}")

    print("Initialized Connection to MT5 Terminal.....")

    mt5.symbol_select(symbol,True)

    raw_rates = mt5.copy_rates_from_pos(symbol,timeframe,0,num_bars)

    mt5.shutdown()
    print("MT5 Terminal Closed Successfully!")

    df = pd.DataFrame(raw_rates)

    df['time'] = pd.to_datetime(df['time'],unit='s')
    df.set_index('time',inplace=True)

    essential_columns = ['open', 'high', 'low', 'close', 'tick_volume']
    return df[essential_columns]

aw_eurusd_df =  connect_and_fetch_forex_data("EURUSD",mt5.TIMEFRAME_H1,5000)
print(f"Matrix Shape Extracted: {aw_eurusd_df.shape}")

Initialized Connection to MT5 Terminal.....
MT5 Terminal Closed Successfully!
Matrix Shape Extracted: (5000, 5)


# **Step 2 :** Feature Enginering, here I am gonna take the raw rates and then create the Engineered stationary features.

In [ ]:
def engineer_stationary_features(raw_df: pd.DataFrame) -> pd.DataFrame: 
    matrix = raw_df.copy()

    matrix['log_return'] = np.log(matrix['close']/matrix['close'].shift(1))

    matrix['high_low_span'] = matrix['high'] - matrix['low']
    matrix['volatility_atr'] = matrix['high_low_span'].rolling(window=14).mean()

    matrix['raw_hour'] = matrix.index.hour
    
    matrix['hour_sin'] = np.sin(2 * np.pi * matrix['raw_hour'] / 24.0)
    matrix['hour_cos'] = np.cos(2 * np.pi * matrix['raw_hour'] / 24.0)

    matrix.dropna(inplace=True)

    final_features = matrix.drop(columns=['open','high','low','raw_hour','high_low_span'])

    return final_features 

final_feature_matrix = engineer_stationary_features(raw_eurusd_df)
print("Pipeline complete! Displaying top 3 rows of processed ML input matrix:")
print(final_feature_matrix[['close', 'log_return', 'volatility_atr', 'hour_sin', 'hour_cos']].head(3))